# Lilly — the full speech instrument (large-v3 against the shipped listener, 925 clips)

**Job:** `speech-instrument`. Pre-registered in `training/PREREGISTRATION.md`,
"v3 — speech — the full instrument, written before it has scored anything".
**Read that section before launching. It says this is the LAST look.**

**What this run does.** Scores two listeners on every clip of the FLEURS bs_ba
test split — 925 clips, all 349 distinct sentences — in one process, twice:
1. `--decode app` — the product's own path (beam 5, this project's normaliser).
   These are the **gate rows**: word error, Bosnian term recall, Croatian
   substitution, with the paired bootstrap over sentences.
2. `--decode rubric` — greedy at temperature 0, WER after Whisper's
   `BasicTextNormalizer`. This is `training/RUBRIC.md`'s definition of the speech
   score, and it has never been computed for any listener.

**What it does NOT do.** It trains nothing and ships nothing. The artefact is a
small zip of JSON and the tee. Nothing here changes `models/lilly/`.

**Attach, before Save & Run All** (`scripts/kaggle_train.py speech-instrument`
does this): **Add data → Datasets → `lilly-listen-large-v3`** — whisper-large-v3,
the candidate, exactly the build half-2 packaged on 1 September (its `model.bin`
is byte-identical to the one the gate scored), unpacked. The half-2 kernel's own
Output (`lilly-listen.zip`) is accepted too if it can still be attached; on
7 September it could not, and the launcher now refuses a push whose source
Kaggle cannot see instead of letting the run die here.

The shipped listener (whisper-small) comes from the published bundle on Hugging
Face. Internet must be **On**.

In [ ]:
# 1. Stop here unless the machine is actually set up
import json, os, shutil, subprocess, sys, urllib.error, urllib.request, zipfile
from pathlib import Path

import torch
assert torch.cuda.is_available(), (
    "No GPU. Right panel -> Session options -> Accelerator -> GPU, then Save & Run All again.")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
GPU = torch.cuda.get_device_name(0)
print(torch.cuda.device_count(), "GPU(s) visible, using:", GPU)

def reachable(url):
    try:
        urllib.request.urlopen(url, timeout=20).close()
    except urllib.error.HTTPError:
        pass
    except Exception as exc:
        raise SystemExit(f"Cannot reach {url} ({exc}). Right panel -> Session options -> Internet -> On.")
for host in ("https://github.com", "https://pypi.org", "https://huggingface.co",
             "https://datasets-server.huggingface.co"):
    reachable(host)
print("network ok")

# The child's stdout is NOT the Kaggle log. Tee everything into Output so a run
# that printed no WER line cannot be mistaken for one that did.
TEE = Path("/kaggle/working/stdout.txt")
TEE.parent.mkdir(parents=True, exist_ok=True)

def run(*cmd, quiet=False, env=None):
    line = "$ " + " ".join(str(c) for c in cmd)
    print(line, flush=True)
    with TEE.open("a", encoding="utf-8") as sink:
        sink.write(line + "\n")
        child = subprocess.Popen([str(c) for c in cmd], stdout=subprocess.PIPE,
                                 stderr=subprocess.STDOUT, text=True, bufsize=1,
                                 env={**os.environ, **(env or {})})
        for out in child.stdout:
            if not quiet:
                print(out, end="", flush=True)
            sink.write(out)
        code = child.wait()
    if code:
        raise subprocess.CalledProcessError(code, cmd)

In [ ]:
# 2. Get the Lilly code — into scratch, never into Output
SCRATCH = Path("/kaggle/temp") if Path("/kaggle/temp").is_dir() else Path("/tmp")
CLONE = SCRATCH / "Lilly"
subprocess.run(["rm", "-rf", str(CLONE)], check=True)
os.chdir(SCRATCH)
run("git", "clone", "-q", "https://github.com/ssaaffaakk/Lilly.git")
assert (CLONE / "training").is_dir(), "clone produced nothing"
os.chdir(CLONE)
print("working in", os.getcwd())

sys.path.insert(0, str(CLONE / "training"))
from kaggle_offload import Offload
OFF = Offload("speech-instrument", os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "manual"))
OFF.hardware(GPU)
print("offload log started:", OFF.body["git"])

In [ ]:
# 3. Install what we need (~2 min). Pins from requirements.txt, not a second list.
NEEDED = ["faster-whisper", "transformers", "soundfile", "huggingface_hub", "pyarrow"]
pins = {}
for line in Path("requirements.txt").read_text(encoding="utf-8").splitlines():
    line = line.split("#")[0].strip()
    if "==" in line:
        pins[line.split("==")[0].strip().lower()] = line
print("pinned here:", [pins[n] for n in NEEDED if n in pins])
run(sys.executable, "-m", "pip", "install", "-q", *[pins.get(n, n) for n in NEEDED])
import faster_whisper, ctranslate2
print("faster-whisper", faster_whisper.__version__, "| ctranslate2", ctranslate2.__version__,
      "| cuda types:", ctranslate2.get_supported_compute_types("cuda"))

In [ ]:
# 4. The clips — all 925 of the FLEURS bs_ba test split, and a hard count
# download_speech_data.py names clips by parquet row order, so this test.tsv is
# the same file, in the same order, as the one on the Mac.
run(sys.executable, "data/scripts/download_speech_data.py", "--split", "test")
TSV = CLONE / "data" / "speech" / "test.tsv"
n_rows = sum(1 for _ in TSV.open(encoding="utf-8"))
print("test.tsv rows:", n_rows)
if n_rows != 925:
    raise SystemExit(f"FLEURS bs_ba test came back as {n_rows} clips, not 925 — "
                     "the pre-registered instrument is all of them; not scoring a partial split")
OFF.metric("clips", n_rows, stage="data")

In [ ]:
# 5. The two listeners
# (a) the SHIPPED one: whisper-small, from the published bundle. fetch_models.py
#     pulls listen/ (and the other parts) from Safak11/lilly.
run(sys.executable, "scripts/fetch_models.py")
SHIPPED = CLONE / "models" / "lilly" / "listen"
assert (SHIPPED / "model.bin").is_file(), "published bundle has no listen/model.bin"
shipped_built = json.loads((SHIPPED / "built.json").read_text()) if (SHIPPED / "built.json").is_file() else {}
print("shipped listener:", shipped_built, f"{(SHIPPED / 'model.bin').stat().st_size / 1e6:.0f} MB")
# The bundle's listen/ IS the shipped small model. Move it aside under the name
# the gate used, so the two labels in every table are the two the owner has
# been reading all week.
PREV = CLONE / "models" / "lilly" / "listen-previous"
SHIPPED.rename(PREV)

# (b) the CANDIDATE: whisper-large-v3. Two shapes are accepted, because the
#     half-2 kernel's Output stopped being attachable on 7 September ("Permission
#     'kernels.get' was denied"; the push said "not valid kernel sources" and
#     version 2 died here):
#     - Kernel output -> lilly-speech-half2: /kaggle/input/**/lilly-listen.zip
#     - Dataset lilly-listen-large-v3 (kaggle_train.py push_listen_candidate):
#       the same build unpacked, built.json beside model.bin. Its model.bin is
#       byte-identical to the one the gate scored. Kaggle extracts archives it is
#       handed as dataset files, which is why the zip is not the only name searched.
#     Either way, built.json is checked below before anything is scored.
INPUT = Path("/kaggle/input")
CAND = CLONE / "models" / "lilly" / "listen"
hits = sorted(INPUT.rglob("lilly-listen.zip")) if INPUT.is_dir() else []
if hits:
    zpath = hits[0]
    print("candidate zip:", zpath, f"{zpath.stat().st_size / 1e6:.0f} MB")
    extract = SCRATCH / "cand-extract"
    subprocess.run(["rm", "-rf", str(extract)], check=True)
    with zipfile.ZipFile(zpath) as zf:
        zf.extractall(extract)
    CAND_SRC = extract / "models" / "lilly" / "listen"
    assert (CAND_SRC / "model.bin").is_file(), f"zip unpacked but no listener at {CAND_SRC}"
    CAND_SRC.rename(CAND)
else:
    dirs = sorted(p.parent for p in INPUT.rglob("built.json") if (p.parent / "model.bin").is_file()) if INPUT.is_dir() else []
    if not dirs:
        raise SystemExit("no candidate attached: neither lilly-listen.zip (Add data -> Kernel output -> "
                         "lilly-speech-half2) nor an unpacked listener with built.json + model.bin "
                         "(Add data -> Datasets -> lilly-listen-large-v3). Relaunch with "
                         "python3 scripts/kaggle_train.py speech-instrument, which attaches the dataset.")
    if len(dirs) > 1:
        raise SystemExit(f"more than one listener attached; refusing to guess which is the candidate: {dirs}")
    print("candidate dir:", dirs[0], f"{(dirs[0] / 'model.bin').stat().st_size / 1e6:.0f} MB")
    # /kaggle/input is read-only and a separate mount, so copy it into the clone,
    # where both listeners sit under models/lilly/ exactly as they do on the Mac.
    shutil.copytree(dirs[0], CAND)
cand_built = json.loads((CAND / "built.json").read_text()) if (CAND / "built.json").is_file() else {}
print("candidate listener:", cand_built, f"{(CAND / 'model.bin').stat().st_size / 1e6:.0f} MB")

# Identity check before anything is scored: the candidate must be the large-v3
# the gate refused, and the shipped one must be the small model it kept.
if cand_built.get("base") != "openai/whisper-large-v3":
    raise SystemExit(f"candidate built.json says {cand_built} — not whisper-large-v3; wrong zip")
if "small" not in str(shipped_built.get("base", "")) and (PREV / "model.bin").stat().st_size > 400_000_000:
    raise SystemExit(f"shipped listener is not whisper-small: {shipped_built}")

In [ ]:
# 6. Smoke: 8 clips through both listeners on the GPU, before the real run
# app.speech puts the listener on CPU int8 by default -- the product's path.
# The rubric's number is about the weights, not the chip, and the same decode
# on a T4 in float16 is what makes 925 clips x 2 listeners x 2 decodes fit in
# a session. Opt-in by environment; the transcripts are cached by weight
# fingerprint so the label under each number is the weights, not the machine.
GPU_ENV = {"LILLY_SPEECH_DEVICE": "cuda", "LILLY_SPEECH_COMPUTE": "float16",
           "LILLY_IGNORE_GUARD": "1", "PYTHONUNBUFFERED": "1"}
run(sys.executable, "training/speech_bench.py", "--clips", "all", "--limit", "8",
    "--model", str(PREV), "--model", str(CAND),
    "--json", "/kaggle/temp/smoke.json", env=GPU_ENV)
# Not --fresh: that empties the cache dict and the next save would overwrite
# bench/speech/.outputs.json with eight clips, throwing away the 200 the gate
# already transcribed. Eight clips cached under the real keys are simply the
# first eight of the real run.
smoke = json.loads(Path("/kaggle/temp/smoke.json").read_text())
assert smoke["n_clips"] == 8 and set(smoke["listeners"]) == {"listen-previous", "listen"}, smoke
print("smoke ok — the path works; those numbers decide nothing")

In [ ]:
# 7. THE GATE ROWS — the product's path, all 925 clips, both listeners, one process
# --clips all is fixed by the pre-registration. Not distinct, not first200.
GATE_JSON = Path("/kaggle/working/speech-gate.json")
run(sys.executable, "training/speech_bench.py", "--clips", "all",
    "--model", str(PREV), "--model", str(CAND),
    "--json", str(GATE_JSON), env=GPU_ENV)
gate = json.loads(GATE_JSON.read_text())
assert gate["decode"] == "app" and gate["clips"] == "all", gate["decode"]
for k, v in gate["listeners"].items():
    OFF.metric(f"gate_wer_{k}", v["wer"], stage="gate")
    OFF.metric(f"gate_recall_{k}", v["term_recall"], stage="gate")
    OFF.metric(f"gate_croatian_{k}", v["croatian"], stage="gate")

In [ ]:
# 8. THE RUBRIC WER — greedy, temperature 0, Whisper's normaliser, all 925 clips
# training/RUBRIC.md's definition. Never computed before, for anything.
RUBRIC_JSON = Path("/kaggle/working/speech-rubric.json")
run(sys.executable, "training/speech_bench.py", "--clips", "all", "--decode", "rubric",
    "--model", str(PREV), "--model", str(CAND),
    "--json", str(RUBRIC_JSON), env=GPU_ENV)
rubric = json.loads(RUBRIC_JSON.read_text())
assert rubric["decode"] == "rubric", rubric["decode"]
for k, v in rubric["listeners"].items():
    OFF.metric(f"rubric_wer_{k}", v["wer"], stage="rubric")

In [ ]:
# 9. The gate, read once, then the package. Nothing is zipped before this passes.
OFF.check_trainproof(TEE)
for rec, name in ((gate, "gate"), (rubric, "rubric")):
    if rec["n_clips"] != 925 or rec["n_clips_in_split"] != 925:
        raise SystemExit(f"{name}: scored {rec['n_clips']} of {rec['n_clips_in_split']} clips, not 925")
    for k, v in rec["listeners"].items():
        if int(v["words"]) < 1000:
            raise SystemExit(f"{name}/{k}: WER over {v['words']} reference words — not a run")
prev, cand = gate["listeners"]["listen-previous"], gate["listeners"]["listen"]
p = gate["paired"]
rows = [
    ("word error, strictly below",        prev["wer"],         cand["wer"],         cand["wer"] < prev["wer"]),
    ("term recall, not below",            prev["term_recall"], cand["term_recall"], cand["term_recall"] >= prev["term_recall"]),
    ("Croatian substitution, not above",  prev["croatian"],    cand["croatian"],    cand["croatian"] <= prev["croatian"]),
]
verdict = all(ok for *_, ok in rows)
lines = ["# The full speech instrument — 925 clips, both listeners, one run",
         "", f"Kaggle, {GPU}. Pre-registered: PREREGISTRATION.md, 'v3 — speech — the full instrument'.",
         "", "## Gate rows (the product's path, --decode app)", "",
         "| threshold | listen-previous (small) | listen (large-v3) | |", "|---|---|---|---|"]
for name, a, b, ok in rows:
    lines.append(f"| {name} | {a:.1f}% | {b:.1f}% | {'pass' if ok else '**FAIL**'} |")
lines += ["", f"Croatian: {cand['croatian_subs']} of {cand['croatian_decided']} decided "
              f"({cand['croatian_targets']} targets) against {prev['croatian_subs']} of {prev['croatian_decided']}; "
              f"paired bootstrap p = {p['croatian']['p']:.4f}.",
          f"Term recall: {p['term_recall']['second_minus_first']:+.1f} points, p = {p['term_recall']['p']:.4f}.",
          "", "## Rubric WER (greedy, temperature 0, BasicTextNormalizer, 925 clips)", "",
          "| listener | WER | words |", "|---|---|---|"]
for k, v in rubric["listeners"].items():
    lines.append(f"| {k} | {v['wer']:.1f}% | {v['words']} |")
lines += ["", f"**Verdict by 'Both, not either': {'SHIPS' if verdict else 'DOES NOT SHIP — and by rule 3 of the pre-registration, large-v3 is closed.'}**"]
report = "\n".join(lines) + "\n"
Path("/kaggle/working/speech-instrument.md").write_text(report, encoding="utf-8")
print(report)
OFF.metric("verdict_ships", bool(verdict), stage="gate")

ZIP = Path("/kaggle/working/lilly-speech-instrument.zip")
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(GATE_JSON, "speech-gate.json")
    z.write(RUBRIC_JSON, "speech-rubric.json")
    z.write("/kaggle/working/speech-instrument.md", "speech-instrument.md")
    z.write(CLONE / "bench" / "speech" / ".outputs.json", "bench-speech-outputs.json")
size = ZIP.stat().st_size
if size < 20_000:
    raise SystemExit(f"zip is {size} bytes — that is not a result")
OFF.finish("complete", [ZIP.name])
out = list(Path("/kaggle/working").rglob("*"))
print(f"Output holds {len(out)} entries, zip {size/1024:.0f} KB")
assert len(out) < 50, [str(x) for x in out[:50]]

**Status ERROR or CANCEL → do not use these numbers.** Read `stdout.txt`, fix the
cause, relaunch. Recovery is not success.

**On COMPLETE:** `scripts/kaggle_train.py speech-instrument --fetch`, unzip:
`speech-gate.json` and `speech-rubric.json` go to `training/speech-instrument/`,
`bench-speech-outputs.json` **replaces** `bench/speech/.outputs.json` (the
transcription cache, now holding all 925 clips for both listeners), and
`speech-instrument.md` is the raw report. Then write the outcome into
`RESULTS-speech.md` and the pre-registration **whichever way it fell**.

If the verdict is DOES NOT SHIP: rule 3 applies. large-v3 is closed. No other
split, no other normaliser, no third instrument.